# Olist E-Commerce Project Starter

Use this starter notebook for the BAN 6003 final project option: **Olist Brazilian E-Commerce**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset is relational. Your main job is to preserve the row meaning of your final ABT.

- A common final ABT is **one row per order**, identified by `order_id`.
- Aggregate `order_items`, `order_payments`, and `order_reviews` to `order_id` before joining them to `orders`.
- Join `products`, category translations, and sellers to item-level data before aggregating those fields to order level.
- `customer_id` identifies the customer record attached to one order; `customer_unique_id` may repeat because one person can place multiple orders.
- Check row counts and duplicate `order_id` values after every major merge.
- Define targets only where the outcome is observable. An undelivered order is not automatically an on-time order, and a missing review is not a low review.
- State when a prediction would be made, then exclude fields that would not be available at that time.


## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [ ]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [ ]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))

In [ ]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

### Key and Granularity Audit

Confirm the expected key for each table before planning joins. A duplicate `order_id` is expected in item, payment, and review detail tables because those tables can contain several records for one order. Their more detailed keys should still be checked. The provided geolocation table is already summarized to one row per zip-code prefix.


In [ ]:
key_audit = pd.DataFrame([
    {"table": "orders", "expected_key": "order_id", "duplicate_keys": orders["order_id"].duplicated().sum(), "missing_key_values": orders["order_id"].isna().sum()},
    {"table": "customers", "expected_key": "customer_id", "duplicate_keys": customers["customer_id"].duplicated().sum(), "missing_key_values": customers["customer_id"].isna().sum()},
    {"table": "order_items", "expected_key": "order_id + order_item_id", "duplicate_keys": items.duplicated(["order_id", "order_item_id"]).sum(), "missing_key_values": items[["order_id", "order_item_id"]].isna().any(axis=1).sum()},
    {"table": "order_payments", "expected_key": "order_id + payment_sequential", "duplicate_keys": payments.duplicated(["order_id", "payment_sequential"]).sum(), "missing_key_values": payments[["order_id", "payment_sequential"]].isna().any(axis=1).sum()},
    {"table": "order_reviews", "expected_key": "review record (review_id may repeat)", "duplicate_keys": reviews.duplicated().sum(), "missing_key_values": reviews["order_id"].isna().sum()},
    {"table": "products", "expected_key": "product_id", "duplicate_keys": products["product_id"].duplicated().sum(), "missing_key_values": products["product_id"].isna().sum()},
    {"table": "sellers", "expected_key": "seller_id", "duplicate_keys": sellers["seller_id"].duplicated().sum(), "missing_key_values": sellers["seller_id"].isna().sum()},
    {"table": "geolocation_zip_prefix", "expected_key": "geolocation_zip_code_prefix", "duplicate_keys": geo["geolocation_zip_code_prefix"].duplicated().sum(), "missing_key_values": geo["geolocation_zip_code_prefix"].isna().sum()},
    {"table": "translations", "expected_key": "product_category_name", "duplicate_keys": translations["product_category_name"].duplicated().sum(), "missing_key_values": translations["product_category_name"].isna().sum()},
])

display(key_audit)
print(f"Item rows sharing an order_id: {items['order_id'].duplicated().sum():,}")
print(f"Payment rows sharing an order_id: {payments['order_id'].duplicated().sum():,}")
print(f"Review rows sharing an order_id: {reviews['order_id'].duplicated().sum():,}")
print(f"Repeated customer_unique_id values: {customers['customer_unique_id'].duplicated().sum():,}")


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Olist notes:

- Keep a table inventory: what does one row mean in each file?
- `order_items`, `order_payments`, and `order_reviews` are not guaranteed to be one row per order.
- Missing review comments are common and should be described, not automatically deleted.
- Product categories may need translation using `product_category_name_translation.csv`.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
- Why this question matters:
- Tables/files used:
- Row meaning of each important table:
- Early data quality issues:
- Initial cleaning plan:
- Possible target/outcome:
- Ethics, privacy, or governance concerns:



In [ ]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.



In [ ]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))

## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what key defines one row, and which tables were joined or aggregated.

Critical integration reminders:

- Enrich item records with product, category, and seller attributes while they are still item-level.
- Aggregate item, payment, and review records separately to `order_id`.
- Join each order-level summary to `orders` with merge validation.
- Treat the order-level ABT as trustworthy only after its row count and unique key have been checked.

The starter below demonstrates this sequence. Revise the variables to fit your question, and explain why each retained variable is useful.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [ ]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



In [ ]:
products_enriched = products.merge(
    translations,
    on="product_category_name",
    how="left",
    validate="many_to_one",
)

item_detail = (
    items
    .merge(
        products_enriched[[
            "product_id", "product_category_name_english", "product_weight_g"
        ]],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        sellers[["seller_id", "seller_state"]],
        on="seller_id",
        how="left",
        validate="many_to_one",
    )
)

item_summary = (
    item_detail.groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        category_count=("product_category_name_english", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        item_price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        product_weight_mean_g=("product_weight_g", "mean"),
    )
    .reset_index()
)

payment_summary = (
    payments.groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "count"),
        payment_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique"),
    )
    .reset_index()
)

reviews_for_summary = reviews.assign(
    review_comment_present=reviews["review_comment_message"].notna().astype(int)
)
review_summary = (
    reviews_for_summary.groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        review_score_mean=("review_score", "mean"),
        review_comment_present=("review_comment_present", "max"),
    )
    .reset_index()
)

customer_geo = customers.merge(
    geo[[
        "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
        "geolocation_records"
    ]],
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng",
    "geolocation_records": "customer_geo_source_records",
})

abt = (
    orders
    .merge(customer_geo, on="customer_id", how="left", validate="one_to_one")
    .merge(item_summary, on="order_id", how="left", validate="one_to_one")
    .merge(payment_summary, on="order_id", how="left", validate="one_to_one")
    .merge(review_summary, on="order_id", how="left", validate="one_to_one")
)

abt["purchase_month"] = abt["order_purchase_timestamp"].dt.month
abt["purchase_dayofweek"] = abt["order_purchase_timestamp"].dt.dayofweek
abt["promised_delivery_days"] = (
    abt["order_estimated_delivery_date"] - abt["order_purchase_timestamp"]
).dt.days
abt["delivery_days"] = (
    abt["order_delivered_customer_date"] - abt["order_purchase_timestamp"]
).dt.days

delivery_outcome_observed = (
    abt["order_status"].eq("delivered")
    & abt["order_delivered_customer_date"].notna()
    & abt["order_estimated_delivery_date"].notna()
)
abt["delivered_late"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
abt.loc[delivery_outcome_observed, "delivered_late"] = (
    abt.loc[delivery_outcome_observed, "order_delivered_customer_date"]
    > abt.loc[delivery_outcome_observed, "order_estimated_delivery_date"]
).astype(int)

abt["high_review_score"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
review_outcome_observed = abt["review_score_mean"].notna()
abt.loc[review_outcome_observed, "high_review_score"] = (
    abt.loc[review_outcome_observed, "review_score_mean"] >= 4
).astype(int)


In [ ]:
abt_validation = pd.Series({
    "orders_before_merges": len(orders),
    "abt_rows_after_merges": len(abt),
    "unique_order_ids": abt["order_id"].nunique(),
    "duplicate_order_ids": abt["order_id"].duplicated().sum(),
    "orders_with_item_summary": abt["order_item_count"].notna().sum(),
    "orders_with_payment_summary": abt["payment_total"].notna().sum(),
    "orders_with_review_summary": abt["review_score_mean"].notna().sum(),
    "orders_with_observed_delivery_target": abt["delivered_late"].notna().sum(),
    "orders_with_observed_review_target": abt["high_review_score"].notna().sum(),
})

display(abt_validation.to_frame("value"))
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the order-level ABT row count, key uniqueness, and target eligibility without creating another permanent database file.


In [ ]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("olist_order_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        '''
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT order_id) AS unique_order_ids,
            SUM(CASE WHEN delivered_late IS NOT NULL THEN 1 ELSE 0 END) AS delivery_target_rows,
            SUM(CASE WHEN high_review_score IS NOT NULL THEN 1 ELSE 0 END) AS review_target_rows
        FROM olist_order_abt
        ''',
        connection,
    )

display(sql_abt_check)


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [ ]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminders:

- If you model `delivered_late`, use only rows where the nullable target is defined; undelivered orders do not reveal a normal late/on-time outcome.
- For a prediction made at purchase time, do not use actual delivery duration, final order status, review fields, or other post-purchase outcomes as predictors.
- If you model `high_review_score`, use only rows with an observed review and do not use review-derived fields as predictors.
- Check class balance, keep a reproducible train/test split, and interpret precision and recall when the positive class is uncommon.
- Explain the intended decision and prediction timing before choosing features.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [ ]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "olist_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "olist_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "olist_model_metrics.csv"
# report_path = REPORTS_DIR / "olist_final_report.html"


In [ ]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "olist_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "olist_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
